In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, MACCSkeys, rdMolDescriptors
from rdkit import DataStructs
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression, mutual_info_regression
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
import scipy.stats as stats
from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning
import os

# Suppress warnings and set up progress bar
warnings.filterwarnings('ignore')
tqdm.pandas()
np.random.seed(42)

# --- 1. Load and clean data ---
# Ensure the file exists before attempting to read it
if not os.path.exists('D2.csv'):
    # Create a dummy D2.csv for demonstration if it doesn't exist
    dummy_data = {
        'Ligand SMILES': ['CCO', 'CCC', 'C(C(=O)O)N', 'C1CCCCC1', 'CC(=O)OC'],
        'IC50 (nM)': ['10.5', '<5', '500', '150', '25.3']
    }
    pd.DataFrame(dummy_data).to_csv('D2.csv', index=False)

df = pd.read_csv('D2.csv')
df = df[['Ligand SMILES', 'IC50 (nM)']].rename(columns={'Ligand SMILES': 'SMILES'})

def clean_ic50(x):
    try:
        x_clean = str(x).strip().replace('<', '').replace('>', '')
        val = float(x_clean)
        return val if val > 0 else None
    except:
        return None

df['IC50 (nM)'] = df['IC50 (nM)'].apply(clean_ic50)
df = df.dropna(subset=['IC50 (nM)']).reset_index(drop=True)
df['pIC50'] = -np.log10(df['IC50 (nM)'] * 1e-9)
df['mol'] = df['SMILES'].progress_apply(lambda s: Chem.MolFromSmiles(s))
df = df.dropna(subset=['mol']).reset_index(drop=True)

# --- 2. Molecular descriptors ---
def mol_descriptors(mol):
    desc_vals = [func(mol) for _, func in Descriptors.descList]
    try:
        MW = Descriptors.MolWt(mol)
        LogP = Descriptors.MolLogP(mol)
        TPSA = Descriptors.TPSA(mol)
        HBA = Descriptors.NumHAcceptors(mol)
        HBD = Descriptors.NumHDonors(mol)
        ROTB = Descriptors.NumRotatableBonds(mol)
    except:
        MW = LogP = TPSA = HBA = HBD = ROTB = 0.0
    return desc_vals + [MW, LogP, TPSA, HBA, HBD, ROTB]

desc_names_basic = [name for name, _ in Descriptors.descList]
desc_physchem_names = ['MolWt', 'LogP', 'TPSA', 'NumHAcceptors', 'NumHDonors', 'NumRotatableBonds']
desc_names = desc_names_basic + desc_physchem_names
desc_data = np.array(df['mol'].progress_apply(mol_descriptors).to_list())
desc_df = pd.DataFrame(desc_data, columns=desc_names).fillna(0)

# --- 3. Fingerprint conversion helper ---
def fp2arr(fp, nbits):
    if not isinstance(fp, DataStructs.ExplicitBitVect):
        if fp is None: return np.zeros(nbits, dtype=np.int8)
        explicit_fp = DataStructs.ExplicitBitVect(nbits)
        for bit in fp.GetOnBits():
            explicit_fp.SetBit(bit)
        fp = explicit_fp
    arr = np.zeros(nbits, dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fps_def = [
    ('morgan2', lambda m: AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048), 2048),
    ('maccs', lambda m: MACCSkeys.GenMACCSKeys(m), 167),
    ('topotorsion', lambda m: rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(m, nBits=2048), 2048),
]

fps_data = []
for name, ffunc, nbits in fps_def:
    arrs = np.stack(df['mol'].progress_apply(lambda m: fp2arr(ffunc(m), nbits)))
    fps_data.append(pd.DataFrame(arrs, columns=[f"{name}_{i}" for i in range(nbits)]))

# --- 4. Assemble features ---
X_all = pd.concat([desc_df] + fps_data, axis=1).fillna(0)
y = df['pIC50'].values

# --- 5. Train/test split (FIRST) ---
# FIX: The train/test split must happen before any data-dependent operations.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

# --- 6. Feature selection (on training data only) ---
print("Performing feature selection on training data...")
selector_var = VarianceThreshold(0.01)
X_train_var = selector_var.fit_transform(X_train_raw)
X_test_var = selector_var.transform(X_test_raw)
feat_names_var = np.array(X_train_raw.columns)[selector_var.get_support()]

selector_f = SelectKBest(f_regression, k=256).fit(X_train_var, y_train)
selector_mi = SelectKBest(mutual_info_regression, k=256).fit(X_train_var, y_train)
idx_union = np.unique(np.concatenate([
    selector_f.get_support(indices=True),
    selector_mi.get_support(indices=True)
]))

X_train_selected = pd.DataFrame(X_train_var[:, idx_union], columns=feat_names_var[idx_union])
X_test_selected = pd.DataFrame(X_test_var[:, idx_union], columns=feat_names_var[idx_union])

# --- 7. Scaling and transformation (on training data only) ---
qt = QuantileTransformer(output_distribution='normal', random_state=42)
y_train_trans = qt.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_trans = qt.transform(y_test.reshape(-1, 1)).flatten()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

# --- 8. Base models & hyperparameter grids ---
base_models = {
    'Ridge': Ridge(),
    'SVR': SVR(),
    'RandomForest': RandomForestRegressor(random_state=42, n_jobs=2),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'ExtraTrees': ExtraTreesRegressor(random_state=42, n_jobs=2),
    'MLP': MLPRegressor(random_state=42, max_iter=500),
    'KNN': KNeighborsRegressor()
}

param_grid = {
    'Ridge': {'alpha': stats.reciprocal(1e-3, 100)},
    'SVR': {'C': stats.reciprocal(0.1, 100), 'gamma': ['scale', 'auto'], 'epsilon': stats.uniform(0.01, 0.3)},
    'RandomForest': {'n_estimators': stats.randint(100, 300), 'max_depth': stats.randint(3, 10)},
    'GradientBoosting': {'n_estimators': stats.randint(100, 300), 'learning_rate': stats.uniform(0.01, 0.1), 'max_depth': stats.randint(3, 8)},
    'ExtraTrees': {'n_estimators': stats.randint(100, 300), 'max_depth': stats.randint(3, 10)},
    'MLP': {'hidden_layer_sizes': [(50,), (100,), (50, 50)], 'alpha': stats.reciprocal(1e-5, 1e-2)},
    'KNN': {'n_neighbors': stats.randint(3, 10)}
}

tuned_models = {}
print("Tuning base models...")
for name, model in base_models.items():
    rs = RandomizedSearchCV(model, param_distributions=param_grid[name], n_iter=20, cv=4,
                            scoring='r2', n_jobs=2, random_state=42, verbose=0)
    rs.fit(X_train_scaled, y_train_trans)
    tuned_models[name] = rs.best_estimator_
    print(f"{name}: best CV R2: {rs.best_score_:.4f}")

# --- 9. Boosted models (not tuned here to simplify) ---
boosted_models = {
    'XGB': XGBRegressor(random_state=42, verbosity=0, n_jobs=2),
    'LGBM': LGBMRegressor(random_state=42, n_jobs=2),
    'CatBoost': CatBoostRegressor(random_state=42, thread_count=2, verbose=0, loss_function="Huber:delta=1")
}

estimators = [(name, model) for name, model in tuned_models.items()] + list(boosted_models.items())

# --- 10. Generate OOF predictions (PROPERLY) ---
# FIX: The meta-learner must be trained on OOF predictions to prevent data leakage.
def get_oof_predictions(base_estimators, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_preds = {name: np.zeros(len(y)) for name, _ in base_estimators}
    for train_idx, val_idx in tqdm(kf.split(X), total=n_splits, desc="Generating OOF"):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr = y[train_idx]
        for name, model in base_estimators:
            clone = model.__class__(**model.get_params())
            clone.fit(X_tr, y_tr)
            oof_preds[name][val_idx] = clone.predict(X_val)
    return np.column_stack([oof_preds[name] for name, _ in base_estimators])

print("Generating OOF predictions...")
X_train_np = np.array(X_train_scaled)
y_train_np = np.array(y_train_trans)
oof_train_preds = get_oof_predictions(estimators, X_train_np, y_train_np)

scaler_meta = StandardScaler()
oof_train_scaled = scaler_meta.fit_transform(oof_train_preds)

# --- 11. Meta-learners and tuning ---
meta_learners = {
    'Ridge': Ridge(),
    'Huber': HuberRegressor(),
    'LightGBM': LGBMRegressor(random_state=42, n_jobs=2),
    'CatBoost': CatBoostRegressor(random_state=42, thread_count=2, verbose=0, loss_function="Huber:delta=1")
}

meta_param_grids = {
    'Ridge': {'alpha': stats.reciprocal(1e-3, 10)},
    'Huber': {'epsilon': stats.uniform(1.2, 2.2)},
    'LightGBM': {'n_estimators': stats.randint(50, 150), 'learning_rate': stats.uniform(0.01, 0.09), 'max_depth': stats.randint(3, 8)},
    'CatBoost': {'iterations': stats.randint(100, 200), 'depth': stats.randint(3, 7), 'learning_rate': stats.uniform(0.01, 0.09)}
}

tuned_meta_models = {}
print("Tuning meta-learners...")
for name, meta_model in meta_learners.items():
    param_dist = meta_param_grids.get(name, {})
    print(f"  Tuning meta-learner: {name} ...")
    if param_dist:
        with ignore_warnings(category=ConvergenceWarning):
            rs_meta = RandomizedSearchCV(meta_model, param_distributions=param_dist, n_iter=10, cv=4,
                                         scoring='r2', n_jobs=2, random_state=42, verbose=0)
            rs_meta.fit(oof_train_scaled, y_train_np)
            tuned_meta_models[name] = rs_meta.best_estimator_
    else:
        tuned_meta_models[name] = meta_model.fit(oof_train_scaled, y_train_np)
        
# --- 12. Fit base models on full training data (after tuning) ---
print("Fitting base models on full training set...")
for _, model in tqdm(estimators, desc="Fitting Base Models"):
    model.fit(X_train_scaled, y_train_np)

# --- 13. Generate meta features for test set ---
oof_test_preds = np.column_stack([model.predict(X_test_scaled) for _, model in estimators])
oof_test_scaled = scaler_meta.transform(oof_test_preds)

# --- 14. Meta predictions ---
# FIX: The meta-blender should not be re-trained on test data.
# The meta-learners trained in step 11 should be used directly for prediction.
meta_X_test = np.column_stack([tuned_meta_models[m].predict(oof_test_scaled) for m in tuned_meta_models.keys()])

# --- 15. Final blender and residual modeling (properly) ---
# The final blender should be trained on the meta_X_test generated from the OOF predictions.
# This part of the code was also a form of data leakage.
# Let's use one of the tuned meta-models (e.g., LightGBM) as the final blender.
final_blender_name = 'LightGBM'
final_blender = tuned_meta_models[final_blender_name]
final_blended_pred = final_blender.predict(oof_test_scaled) # Corrected to use the tuned meta-model
final_blended_r2 = r2_score(y_test_trans, final_blended_pred)

# --- 16. Residual modeling (PROPERLY) ---
# FIX: Residual modeling should also be trained on the OOF data from the training set.
# The original code fit the residual model on test data, a clear form of data leakage.
residuals_train = y_train_np - final_blender.predict(oof_train_scaled)
residual_model = Ridge(alpha=1.0)
residual_model.fit(oof_train_scaled, residuals_train)

residuals_pred = residual_model.predict(oof_test_scaled)
final_refined_pred = final_blended_pred + residuals_pred
final_refined_r2 = r2_score(y_test_trans, final_refined_pred)

# --- 17. Reporting ---
meta_r2s = {k: r2_score(y_test_trans, tuned_meta_models[k].predict(oof_test_scaled)) for k in tuned_meta_models}
print("\nMeta-Learner R² scores:")
for k, v in meta_r2s.items():
    print(f"  {k}: {v:.4f}")
print(f"LightGBM blender R² (using a tuned meta-model): {final_blended_r2:.4f}")
print(f"Final residual-refined R²: {final_refined_r2:.4f}")

# --- 18. Visualization ---
plt.figure(figsize=(12, 6))
bars = list(meta_r2s.values()) + [final_blended_r2, final_refined_r2]
labels = list(meta_r2s.keys()) + [f'{final_blender_name} Blender', 'Residual Refined']
colors = ['blue', 'cyan', 'green', 'orange', 'purple', 'red']

plt.bar(labels, bars, color=colors)
plt.axhline(0.75, color='red', linestyle='--', label='Target R² = 0.75')
plt.ylabel('Test R²')
plt.title('Meta-Learners, Nonlinear Blender, and Residual Correction Performance')
plt.legend()
plt.grid(axis='y')
for i, v in enumerate(bars):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

best_name = 'Residual Refined'
best_pred = final_refined_pred

plt.figure(figsize=(6, 6))
plt.scatter(y_test_trans, best_pred, alpha=0.6, edgecolor='k', linewidth=0.5)
lims = [min(y_test_trans.min(), best_pred.min()), max(y_test_trans.max(), best_pred.max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('True pIC50 (scaled)')
plt.ylabel('Predicted pIC50 (scaled)')
plt.title(f"Best model: {best_name}\nTest R² = {final_refined_r2:.4f}")
plt.grid()
plt.tight_layout()
plt.show()

print("Completed: Nonlinear meta-blending + residual modeling applied (data leakage fixed).")